In [ ]:
import sys, os
import compass
print("Using COMPASS version:", compass.__version__)

from compass.utils import plot_embed_with_label
from compass import PreTrainer, FineTuner, loadcompass #, get_minmal_epoch
from compass.utils import plot_embed_with_label,plot_performance, score2
from compass.tokenizer import CANCER_CODE

import os
from tqdm import tqdm
from itertools import chain
import pandas as pd
import numpy as np
import random, torch
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style = 'white', font_scale=1.3)
import warnings
warnings.filterwarnings("ignore")

def onehot(S):
    assert type(S) == pd.Series, 'Input type should be pd.Series'
    dfd = pd.get_dummies(S, dummy_na=True)
    nanidx = dfd[dfd[np.nan].astype(bool)].index
    dfd.loc[nanidx, :] = np.nan
    dfd = dfd.drop(columns=[np.nan])*1.
    cols = dfd.sum().sort_values(ascending=False).index.tolist()
    dfd = dfd[cols]
    return dfd


pth = './compass_run/PT_v100//pretrainer.pt'
pretrainer = loadcompass(pth)
data_path = './data/ITRP/'

df_label = pd.read_pickle(os.path.join(data_path, 'ITRP.PATIENT.TABLE'))
df_tpm = pd.read_pickle(os.path.join(data_path, 'ITRP.TPM.TABLE'))[pretrainer.feature_name]
df_tpm.shape, df_label.shape

dfcx = df_label.cancer_type.map(CANCER_CODE).to_frame('cancer_code').join(df_tpm)

df_task = onehot(df_label.response_label)
size = df_label.groupby('cohort').size()
size = size.index + "\n(n = " + size.astype(str) + ")"
cohorts = df_label.groupby('cohort').size().sort_values().index.tolist()
#cohorts = ['Choueiri']


def leave_one_cohort_out(cohorts):
    # Create a list of lists, each missing one element from the original list
    return [(cohorts[i], cohorts[:i] + cohorts[i+1:]) for i in range(len(cohorts))]
train_test_cohorts = leave_one_cohort_out(cohorts)




params = dict(
    mode='LFT',
    seed=42,
    lr=3e-3,
    device='cuda',
    weight_decay=1e-8,
    batch_size=16,
    max_epochs=100,
    patience = 10,
    task_loss_type="ce_loss",
    task_type="c",
    task_dense_layer=[16],
    task_batch_norms=True,
    task_loss_weight=1,
    entropy_weight=1e-2,
    with_wandb=False,
    save_best_model=False,
    verbose=False,
)





seed = 42
for seed in [24, 42, 64]:

    for mode in ['LFT']: #,
    
        print('Evaludation on Model %s' % mode)
    
        params['mode'] = mode
        params['seed'] = seed
        
        work_dir = './compass_run/FT_v100/LOCO_%s_%s' % (mode, seed)
        if not os.path.exists(work_dir):
            os.makedirs(work_dir)
        
        res = []
        for test_cohort, train_cohorts in train_test_cohorts:
    
            train_cohort_name = 'Leave_%s_out' % test_cohort
            
            ## Get data for this cohort
            cohort_idx = df_label[df_label['cohort'].isin(train_cohorts)].index
            cohort_X = dfcx.loc[cohort_idx]
            cohort_y = df_task.loc[cohort_idx]
    

            ## Get features for specific method
            train_X = cohort_X
            train_y = cohort_y

    
            test_cohort_idx = df_label[df_label['cohort'] == test_cohort].index
            test_cohort_X = dfcx.loc[test_cohort_idx]
            test_cohort_y = df_task.loc[test_cohort_idx]
            
            pretrainer = pretrainer.copy()
            finetuner = FineTuner(pretrainer, **params, 
                                  work_dir= work_dir, 
                                  task_name = '%s' % train_cohort_name)
            
            
            finetuner = finetuner.tune(dfcx_train = train_X,
                                       dfy_train = train_y,
                                       min_mcc=0.8,)  
    
            _, pred_testy = finetuner.predict(test_cohort_X, batch_size = 16)
    
            pred_testy['train_cohort'] = train_cohort_name
            pred_testy['test_cohort'] = test_cohort 
            
            pred_testy['best_epoch'] = finetuner.best_epoch
            pred_testy['n_trainable_params'] = finetuner.count_parameters()
            pred_testy['mode'] = mode
            pred_testy['seed'] = seed
            pred_testy['batch_size'] = params['batch_size']
            pred_testy['task_dense_layer'] = str(params['task_dense_layer'])
            dfp = test_cohort_y.join(pred_testy)
    
            y_true, y_prob, y_pred = dfp['R'], dfp[1], dfp[[0, 1]].idxmax(axis=1)
            fig = plot_performance(y_true, y_prob, y_pred)
            fig.suptitle('cohort to cohort transfer: train: %s, test: %s' % (train_cohort_name, test_cohort), fontsize=16)
            fig.savefig(os.path.join(work_dir, 'CTCT_train_%s_test_%s.jpg' % (train_cohort_name, test_cohort)))
            res.append(dfp)
        
        dfs = pd.concat(res)
        dfp = dfs.groupby(['train_cohort', 'test_cohort']).apply(lambda x:score2(x['R'], x[1], x[[0, 1]].idxmax(axis=1)))
    
        #roc, prc, f1, acc, mcc
        dfp = dfp.apply(pd.Series)
        dfp.columns = ['ROC', 'PRC', 'F1', 'ACC', 'MCC']
        dfp = dfp.reset_index()
        
        dfs.to_csv(os.path.join(work_dir, 'source_performance.tsv'), sep='\t')
        dfp.to_csv(os.path.join(work_dir, 'metric_performance.tsv'), sep='\t')

Using COMPASS version: 2.5
Evaludation on Model LFT


 49%|#######################################2                                        | 49/100 [14:42<15:18, 18.00s/it]


Stopping early at epoch 50. Meet minimal requirements by: f1=0.86,mcc=0.80,prc=0.94, roc=0.97


 52%|#########################################6                                      | 52/100 [15:27<14:16, 17.84s/it]


Stopping early at epoch 53. Meet minimal requirements by: f1=0.87,mcc=0.81,prc=0.94, roc=0.97


...

 51%|########################################8                                       | 51/100 [15:05<14:30, 17.76s/it]


Stopping early at epoch 52. Meet minimal requirements by: f1=0.88,mcc=0.83,prc=0.96, roc=0.98


 58%|##############################################4                                 | 58/100 [17:22<12:35, 17.98s/it]


Stopping early at epoch 59. Meet minimal requirements by: f1=0.87,mcc=0.82,prc=0.96, roc=0.98


 44%|###################################2                                            | 44/100 [12:54<14:59, 16.06s/it]

In [ ]:
pwd

In [ ]:
ls